# Excepciones y estabilidad

Este cuaderno ayuda a priorizar errores por recurrencia y a distinguir entre incidencias continuas y brotes puntuales.

In [ ]:
import sys
from pathlib import Path

NOTEBOOK_CWD = Path.cwd()
candidate_roots = [
    NOTEBOOK_CWD,
    NOTEBOOK_CWD.parent if NOTEBOOK_CWD.name == 'notebooks' else NOTEBOOK_CWD,
    NOTEBOOK_CWD / 'observability' / 'd365-fo-observability',
]
PROJECT_ROOT = None
for candidate in candidate_roots:
    if (candidate / 'src' / 'kql_runner.py').exists() and (candidate / 'queries').exists():
        PROJECT_ROOT = candidate
        break
if PROJECT_ROOT is None:
    raise RuntimeError('No se pudo localizar observability/d365-fo-observability desde el directorio actual')
SRC_DIR = PROJECT_ROOT / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from kql_runner import build_client, load_config, load_kql, plot_bar, plot_timeseries, run_kql

config = load_config()
client = build_client(config=config)
EXCEPTIONS_DAYS = config['query_days']

In [ ]:
df_summary = run_kql(client, load_kql('40_exceptions.kql'), days=EXCEPTIONS_DAYS, name='Exceptions summary', config=config)
df_summary['ExceptionLabel'] = df_summary['ExceptionType'].astype(str) + ' | ' + df_summary['ProblemId'].astype(str)
display(df_summary.head(20))
plot_bar(df_summary, 'ExceptionLabel', 'Occurrences', 'Errores mas recurrentes', top_n=15)

In [ ]:
df_timeseries = run_kql(client, load_kql('41_exceptions_timeseries.kql'), days=EXCEPTIONS_DAYS, name='Exceptions timeseries', config=config)
display(df_timeseries.tail(24))
plot_timeseries(df_timeseries, 'timestamp', ['Exceptions'], 'Evolucion horaria de excepciones')

df_latest = run_kql(client, load_kql('42_latest_exceptions.kql'), days=EXCEPTIONS_DAYS, name='Latest exceptions', config=config)
display(df_latest.head(30))

## Recomendacion de visualizacion

- Ranking por Occurrences para portada o backlog de problemas.
- Serie temporal para detectar regresiones tras despliegues o picos de negocio.
- Latest exceptions como vista de detalle, no como primer widget salvo que el equipo trabaje en modo soporte reactivo.